# COMP90042 v10 — Diagnostic experiments on top of v9

This notebook runs five targeted experiments designed to (a) **diagnose** what is holding back the v9 pipeline, (b) **fix** the most likely culprits at low cost, and (c) produce the analysis material needed for the report's *Critical Analysis* section. It is meant to be run **after** `v9.ipynb` has produced its cached artifacts in `outputs_notebook_v9/`.

The experiments, in order:

**A. Oracle retrieval.** Feed gold evidence to the v9 classifier at inference time. The accuracy that comes back is the ceiling the current classifier could reach if retrieval were perfect. A single number that tells us whether retrieval or classification is the binding constraint.

**B. Reranker fix.** Retrain the cross-encoder with `pos_weight=4` and a lower learning rate. The original training had `best_epoch=1` followed by monotonic dev-F decay, which is the classic signature of BCE overfitting under a 4:1 negative-to-positive ratio.

**C. BM25 × reranker fusion.** Combine the (well-calibrated, lexical) BM25 score with the (semantic) reranker logit through a per-claim normalised weighted sum. Sweep the mixing weight on dev.

**D. LLM cascade for DISPUTED.** Use an open-source 3B model (Qwen2.5-3B-Instruct) in 4-bit to detect whether the retrieved evidence pieces internally contradict; route YES answers to DISPUTED. This addresses the assignment's *hybrid system* recommendation while keeping the LLM's role narrow and defensible.

**E. Per-class error analysis.** Qualitative inspection of canonical error types on dev — exactly the material the report's Critical Analysis section needs.

We finish with a small **variance run** over three seeds for the final configuration, so the report can state mean ± std instead of a single-seed point estimate.

All new outputs land in `outputs_notebook_v10/`, distinct from the v9 directory. Nothing in v9 is overwritten.

## 0. Setup

In [1]:
# Environment setup. The path / device / seed conventions mirror v9
# exactly so cached pickle artifacts deserialise cleanly and downstream
# metrics are directly comparable to v9's reported numbers.
import os
os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from pathlib import Path
import json, pickle, random, time, gc, math
from collections import Counter

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# -------- Paths --------
DATA_DIR   = Path("data")
V9_DIR     = Path("outputs_notebook_v9")    # source of v9 cached artifacts
V10_DIR    = Path("outputs_notebook_v10")   # this notebook's outputs
V9_CACHE   = V9_DIR / "cache"
V9_MODELS  = V9_DIR / "models"
V10_CACHE  = V10_DIR / "cache"
V10_MODELS = V10_DIR / "models"
for d in (V10_DIR, V10_CACHE, V10_MODELS):
    d.mkdir(exist_ok=True, parents=True)

# -------- Reproducibility (same SEED as v9 for cross-comparison) --------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -------- Hardware / autocast policy (same logic as v9) --------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported():
    AUTOCAST_DTYPE, USE_GRAD_SCALER = torch.bfloat16, False
elif DEVICE.type == "cuda":
    AUTOCAST_DTYPE, USE_GRAD_SCALER = torch.float16, True
else:
    AUTOCAST_DTYPE, USE_GRAD_SCALER = torch.float32, False
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark   = True

# -------- Task constants (must match v9) --------
LABELS   = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

RERANKER_MODEL_NAME   = "cross-encoder/ms-marco-MiniLM-L-12-v2"
CLASSIFIER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-12-v2"   # v9's BEST_CLASSIFIER
RERANKER_MAX_LEN, CLASSIFIER_MAX_LEN = 256, 512
BM25_CANDIDATE_K = 500
MIN_FINAL_K, MAX_FINAL_K = 1, 5

print(f"Device: {DEVICE}  autocast_dtype={AUTOCAST_DTYPE}  grad_scaler={USE_GRAD_SCALER}")
print(f"V9 artifacts dir exists: {V9_DIR.exists()}  cache: {V9_CACHE.exists()}  models: {V9_MODELS.exists()}")

Device: cuda  autocast_dtype=torch.bfloat16  grad_scaler=False
V9 artifacts dir exists: True  cache: True  models: True


D:\_Search\_Study\COMP90042-NLP\A3_Group\COMP90042_2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Data and v9 artifacts

We load the official splits exactly as v9 did, then load the three classes of cached artifact that v9 produced: the BM25 top-500 candidate pools (the input to the reranker), the per-claim reranker score caches (the output of v9's reranker), and the final selected evidence per claim (the input to v9's classifier). A sanity check at the bottom confirms we reproduce v9's reported dev retrieval F = 0.2448 — if that fails, something has drifted between v9's run and this one and we must fix it before running any experiment.

In [2]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_claims = load_json(DATA_DIR / "train-claims.json")
dev_claims   = load_json(DATA_DIR / "dev-claims.json")
test_claims  = load_json(DATA_DIR / "test-claims-unlabelled.json")
evidence     = load_json(DATA_DIR / "evidence.json")
evidence_ids = list(evidence.keys())

print(f"train: {len(train_claims):,}   dev: {len(dev_claims):,}   "
      f"test: {len(test_claims):,}   evidence: {len(evidence):,}")

# -------- Metric helpers (mirror eval.py / v9.cell-8) --------
def evidence_f1_for_claim(pred_eids, gold_eids):
    pred_eids = list(pred_eids); gold_eids = list(gold_eids)
    if not pred_eids:
        return 0.0
    pred_set = set(pred_eids)
    correct = sum(1 for eid in gold_eids if eid in pred_set)
    if correct == 0:
        return 0.0
    p = correct / len(pred_eids); r = correct / len(gold_eids)
    return 2 * p * r / (p + r)

def evaluate_retrieval_only(retrieval, gold_claims):
    return float(np.mean([
        evidence_f1_for_claim(retrieval[cid], c["evidences"])
        for cid, c in gold_claims.items()
    ]))

def evaluate_submission(predictions, gold_claims, verbose=True):
    f_scores, correct = [], 0
    for cid, gold in gold_claims.items():
        p = predictions[cid]
        f_scores.append(evidence_f1_for_claim(p["evidences"], gold["evidences"]))
        correct += int(p["claim_label"] == gold["claim_label"])
    F = float(np.mean(f_scores)); A = correct / len(gold_claims)
    H = 0.0 if (F + A) == 0 else 2 * F * A / (F + A)
    if verbose:
        print(f"  F={F:.4f}   A={A:.4f}   H={H:.4f}")
    return {"F": F, "A": A, "H": H}

train: 1,228   dev: 154   test: 153   evidence: 1,208,827


In [3]:
# Load v9 cached artifacts. If any are missing the user must run v9 first.
def must_load(path: Path, what: str):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing v9 artifact: {path}\n"
            f"  Run v9.ipynb end-to-end first; it produces {what} during its run."
        )
    with open(path, "rb") as f:
        return pickle.load(f)

train_bm25 = must_load(V9_CACHE / "train_bm25_top500.pkl", "BM25 top-500 candidate pool")
dev_bm25   = must_load(V9_CACHE / "dev_bm25_top500.pkl",   "BM25 top-500 candidate pool")
test_bm25  = must_load(V9_CACHE / "test_bm25_top500.pkl",  "BM25 top-500 candidate pool")

train_ce_v9 = must_load(V9_CACHE / "train_ce_scores.pkl", "v9 reranker scores")
dev_ce_v9   = must_load(V9_CACHE / "dev_ce_scores.pkl",   "v9 reranker scores")
test_ce_v9  = must_load(V9_CACHE / "test_ce_scores.pkl",  "v9 reranker scores")

train_retrieval_v9 = must_load(V9_CACHE / "train_final_evidence.pkl", "v9 final evidence")
dev_retrieval_v9   = must_load(V9_CACHE / "dev_final_evidence.pkl",   "v9 final evidence")
test_retrieval_v9  = must_load(V9_CACHE / "test_final_evidence.pkl",  "v9 final evidence")

# Sanity: must reproduce v9's reported dev retrieval F of 0.2448.
v9_dev_F = evaluate_retrieval_only(dev_retrieval_v9, dev_claims)
print(f"v9 reference dev retrieval F = {v9_dev_F:.4f}  (v9 reported 0.2448)")
assert abs(v9_dev_F - 0.2448) < 5e-3, (
    "Cached v9 artifacts don't reproduce v9's dev F. "
    "Re-run v9.ipynb to regenerate cache before continuing."
)

# Label distribution from train — used by class-weight code below.
label_dist = Counter(c["claim_label"] for c in train_claims.values())
print("Train label distribution:")
for l in LABELS:
    n = label_dist[l]
    print(f"  {l:<20s} {n:4d} ({100*n/len(train_claims):.2f}%)")

v9 reference dev retrieval F = 0.2448  (v9 reported 0.2448)
Train label distribution:
  SUPPORTS              519 (42.26%)
  REFUTES               199 (16.21%)
  NOT_ENOUGH_INFO       386 (31.43%)
  DISPUTED              124 (10.10%)


## 2. Shared helpers — selection strategies and classifier training

Experiments B, C, and F all need to either re-run v9's relative-δ selection logic on new score caches, or retrain v9's classifier under different conditions. We factor those out into reusable helpers here. The classifier training function is a stripped-down version of v9's (no Muon optimiser, no double-optimiser plumbing — single AdamW since we never observed Muon helping in v9's logs). Loss, learning rate, batch size, and epoch count match v9 exactly.

In [4]:
# -------- Selection strategies (v9.cell-22, generalised for any score key) --------
def select_relative_delta(ce_cache, delta, kmin=MIN_FINAL_K, kmax=MAX_FINAL_K,
                          score_key="ce_logit"):
    out = {}
    for cid, d in ce_cache.items():
        scores = d[score_key]
        order  = np.argsort(-scores)
        eids   = [d["eids"][i] for i in order]
        ordered = scores[order]
        keep = [eids[0]]; top = ordered[0]
        for j in range(1, min(kmax, len(eids))):
            if ordered[j] >= top - delta:
                keep.append(eids[j])
            else:
                break
        out[cid] = keep[:kmax] if len(keep) >= kmin else keep
    return out

def select_fixed_k(ce_cache, k, kmax=MAX_FINAL_K, score_key="ce_logit"):
    out = {}
    for cid, d in ce_cache.items():
        order = np.argsort(-d[score_key])
        out[cid] = [d["eids"][i] for i in order[:min(k, kmax)]]
    return out

In [5]:
# -------- Classifier training & inference helpers (v9.cell-27/28, minimal) --------
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

CLASSIFIER_BATCH_TRAIN = 16
CLASSIFIER_BATCH_EVAL  = 32
CLASSIFIER_EPOCHS      = 4
CLASSIFIER_LR          = 1e-5
WEIGHT_DECAY = 0.001
WARMUP_RATIO = 0.1
GRAD_CLIP    = 1.0


def build_classifier_evidence(cid, claim, retrieval_dict, max_k=MAX_FINAL_K, include_gold=True):
    """Gold ∪ retrieved (gold first), deduped, capped at max_k. Same logic as v9."""
    gold = list(claim.get("evidences", [])) if include_gold else []
    retrieved = list(retrieval_dict.get(cid, []))
    seen, chosen = set(), []
    for eid in gold + retrieved:
        if eid in seen or eid not in evidence:
            continue
        seen.add(eid); chosen.append(eid)
        if len(chosen) >= max_k:
            break
    if not chosen and retrieved:
        chosen = retrieved[:1]
    return chosen


class ClaimEvidenceDataset(Dataset):
    def __init__(self, claims_dict, retrieval_dict, tokenizer, include_gold,
                 max_len=CLASSIFIER_MAX_LEN):
        self.cids = list(claims_dict.keys())
        self.claims, self.retrieval = claims_dict, retrieval_dict
        self.tokenizer, self.max_len = tokenizer, max_len
        self.include_gold = include_gold
        self.has_label = "claim_label" in next(iter(claims_dict.values()))

    def __len__(self): return len(self.cids)

    def __getitem__(self, i):
        cid = self.cids[i]; c = self.claims[cid]
        ev_ids = build_classifier_evidence(cid, c, self.retrieval,
                                            include_gold=self.include_gold)
        if not ev_ids:
            ev_ids = [next(iter(evidence.keys()))]
        sep = f" {self.tokenizer.sep_token} "
        ev_text = sep.join(evidence[e] for e in ev_ids)
        enc = self.tokenizer(c["claim_text"], ev_text, truncation=True,
                              padding="max_length", max_length=self.max_len,
                              return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.has_label:
            item["labels"] = torch.tensor(LABEL2ID[c["claim_label"]], dtype=torch.long)
        return item


def make_loader(claims_dict, retrieval_dict, tokenizer, batch_size, include_gold, shuffle=False):
    ds = ClaimEvidenceDataset(claims_dict, retrieval_dict, tokenizer, include_gold)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                    pin_memory=(DEVICE.type=="cuda"), num_workers=0)
    return ds, dl


def fresh_classifier(model_name=CLASSIFIER_MODEL_NAME):
    return AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    ).to(DEVICE)


@torch.inference_mode()
def predict_labels(model, loader, dataset):
    model.eval()
    preds = []
    for batch in loader:
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items() if k != "labels"}
        with torch.autocast(device_type="cuda" if DEVICE.type=="cuda" else "cpu",
                            dtype=AUTOCAST_DTYPE, enabled=(DEVICE.type=="cuda")):
            logits = model(**batch).logits
        preds.extend(logits.float().argmax(dim=-1).cpu().tolist())
    return {dataset.cids[i]: ID2LABEL[p] for i, p in enumerate(preds)}


def compute_class_weights(power: float) -> torch.Tensor:
    """power=0 → uniform; power=0.5 → sqrt inv-freq; power=1 → inv-freq."""
    counts = np.array([label_dist[l] for l in LABELS], dtype=np.float64)
    if power == 0.0:
        w = np.ones_like(counts)
    else:
        inv = counts.sum() / np.maximum(counts, 1)
        w = inv ** power
    w = w * (len(LABELS) / w.sum())
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


def train_classifier(retrieval_train, retrieval_dev,
                      model_name=CLASSIFIER_MODEL_NAME,
                      weighting_power=0.0, epochs=CLASSIFIER_EPOCHS,
                      seed=SEED, tag=""):
    """Train a fresh classifier. Returns (best_dict, history).
    `best_dict` keys: epoch, F, A, H, state, pred_labels."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = fresh_classifier(model_name)

    _, train_loader = make_loader(train_claims, retrieval_train, tokenizer,
                                   CLASSIFIER_BATCH_TRAIN, include_gold=True, shuffle=True)
    dev_ds, dev_loader = make_loader(dev_claims, retrieval_dev, tokenizer,
                                      CLASSIFIER_BATCH_EVAL, include_gold=False, shuffle=False)

    loss_fn = nn.CrossEntropyLoss(weight=compute_class_weights(weighting_power))
    optim = torch.optim.AdamW(model.parameters(), lr=CLASSIFIER_LR,
                                weight_decay=WEIGHT_DECAY)
    total_steps = len(train_loader) * epochs
    sched = get_linear_schedule_with_warmup(
        optim, num_warmup_steps=int(WARMUP_RATIO * total_steps),
        num_training_steps=total_steps,
    )
    scaler = (torch.amp.GradScaler("cuda", enabled=USE_GRAD_SCALER)
              if hasattr(torch.amp, "GradScaler")
              else torch.cuda.amp.GradScaler(enabled=USE_GRAD_SCALER))

    best = {"epoch": -1, "A": -1.0, "F": 0.0, "H": 0.0, "state": None, "pred_labels": None}
    history = []
    for epoch in range(1, epochs + 1):
        model.train(); losses = []; t0 = time.time()
        for batch in tqdm(train_loader, desc=f"  {tag} ep{epoch}/{epochs}", leave=False):
            batch  = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            labels = batch.pop("labels")
            with torch.autocast(device_type="cuda" if DEVICE.type=="cuda" else "cpu",
                                dtype=AUTOCAST_DTYPE, enabled=(DEVICE.type=="cuda")):
                logits = model(**batch).logits
                loss   = loss_fn(logits, labels)
            if USE_GRAD_SCALER:
                scaler.scale(loss).backward(); scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optim); scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optim.step()
            sched.step(); optim.zero_grad(set_to_none=True)
            losses.append(float(loss.item()))

        pred_labels = predict_labels(model, dev_loader, dev_ds)
        preds = {cid: {"claim_label": pred_labels[cid], "evidences": retrieval_dev[cid]}
                 for cid in dev_claims}
        m = evaluate_submission(preds, dev_claims, verbose=False)
        history.append({"epoch": epoch, "loss": float(np.mean(losses)),
                         **m, "time_s": round(time.time()-t0, 1)})
        print(f"  {tag} ep{epoch}: loss={np.mean(losses):.4f}  F={m['F']:.4f}  "
              f"A={m['A']:.4f}  H={m['H']:.4f}  ({time.time()-t0:.1f}s)")
        if m["A"] > best["A"]:
            best = {"epoch": epoch, **m,
                    "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                    "pred_labels": dict(pred_labels)}
    print(f"  best ep{best['epoch']}: F={best['F']:.4f} A={best['A']:.4f} H={best['H']:.4f}")
    del model
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return best, history

## Experiment A — Oracle retrieval diagnostic

The single most informative number we can produce right now is **the classifier's accuracy when it receives gold evidence**. Call this A_oracle. The v9 classifier under noisy retrieval achieved A = 0.4805. If A_oracle is much higher (say 0.65 or above), the binding constraint is retrieval — the classifier can in fact discriminate when fed correct evidence, but the noisy retrieval starves it of signal. If A_oracle is similar to A_v9 (around 0.50), the binding constraint is the classification problem itself: even with perfect retrieval, 1228 training claims and a 10%-frequency DISPUTED class do not give the classifier enough signal to separate the four labels.

This single experiment dictates where to invest the remaining time: improve retrieval (Experiments B and C), or pivot to a different classification approach (Experiment D's LLM cascade). It is also, by itself, a reportable finding: the rubric values Critical Analysis heavily, and "we identify that X is the binding constraint and demonstrate it via Y" is exactly the kind of claim Soundness rewards.

Concretely: we replace dev retrieval with gold evidence (so retrieval F is trivially 1.0), then evaluate v9's classifier — either by loading its saved weights if available, or by retraining from scratch with identical settings.

In [6]:
# Oracle retrieval = gold evidence per dev claim. Retrieval F is trivially 1.0.
oracle_dev_retrieval = {cid: list(c["evidences"]) for cid, c in dev_claims.items()}
assert abs(evaluate_retrieval_only(oracle_dev_retrieval, dev_claims) - 1.0) < 1e-9

# Try v9's saved classifier weights so we don't retrain. The classifier
# was trained on (gold + retrieved), so feeding it pure gold at inference
# is a fair test — the training distribution covered gold.
v9_clf_path = V9_MODELS / "classifier_C2-minilm-marco_W0.pt"
pred_labels_oracle = None
if v9_clf_path.exists():
    print(f"Loading v9 classifier weights from {v9_clf_path} ...")
    tok_oracle = AutoTokenizer.from_pretrained(CLASSIFIER_MODEL_NAME)
    model_oracle = fresh_classifier()
    model_oracle.load_state_dict(torch.load(v9_clf_path, map_location=DEVICE))
    dev_ds_o, dev_loader_o = make_loader(dev_claims, oracle_dev_retrieval, tok_oracle,
                                          CLASSIFIER_BATCH_EVAL, include_gold=False, shuffle=False)
    pred_labels_oracle = predict_labels(model_oracle, dev_loader_o, dev_ds_o)
    del model_oracle
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
else:
    print(f"No v9 weights at {v9_clf_path}; retraining a fresh classifier "
          f"(same hparams as v9, ~1 min). To skip retraining next time, "
          f"re-run v9.ipynb so it saves the .pt file.")
    best_retrained, _ = train_classifier(
        train_retrieval_v9, oracle_dev_retrieval,
        weighting_power=0.0, epochs=CLASSIFIER_EPOCHS, tag="Oracle-clf",
    )
    pred_labels_oracle = best_retrained["pred_labels"]

# A_oracle = classification accuracy under oracle retrieval.
correct = sum(int(pred_labels_oracle[cid] == c["claim_label"])
              for cid, c in dev_claims.items())
A_oracle = correct / len(dev_claims)

# Reference points for comparison.
A_v9       = 0.4805   # v9's final reported accuracy
A_majority = max(label_dist.values()) / len(train_claims)   # all-SUPPORTS baseline

print("\nDIAGNOSTIC SUMMARY")
print(f"  A_majority (all SUPPORTS) = {A_majority:.4f}")
print(f"  A_v9 (noisy retrieval)    = {A_v9:.4f}")
print(f"  A_oracle (gold evidence)  = {A_oracle:.4f}")
print(f"  Δ (oracle - v9)           = {A_oracle - A_v9:+.4f}")

Loading v9 classifier weights from outputs_notebook_v9\models\classifier_C2-minilm-marco_W0.pt ...


[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1`.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8395.96it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1]) vs model:torch.Size([4])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1, 384]) vs model:torch.Size([4, 384])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.



DIAGNOSTIC SUMMARY
  A_majority (all SUPPORTS) = 0.4226
  A_v9 (noisy retrieval)    = 0.4805
  A_oracle (gold evidence)  = 0.4091
  Δ (oracle - v9)           = -0.0714


In [7]:
# Per-class accuracy under oracle vs v9 — the more informative view.
def per_class_accuracy(pred_labels, claims):
    rows = []
    for lbl in LABELS:
        cids = [cid for cid, c in claims.items() if c["claim_label"] == lbl]
        acc = float(np.mean([pred_labels[cid] == lbl for cid in cids])) if cids else 0.0
        rows.append({"label": lbl, "n": len(cids), "accuracy": acc})
    return pd.DataFrame(rows)

oracle_per_class = per_class_accuracy(pred_labels_oracle, dev_claims)
print("Per-class accuracy under oracle retrieval:")
print(oracle_per_class.to_string(index=False))

# Interpretation thresholds: tunable but the cuts below are deliberately
# conservative. >+0.15 is a clear retrieval-bound signal; <+0.05 is a
# clear classification-bound signal; in between, the system is mixed.
delta = A_oracle - A_v9
if delta > 0.15:
    interpretation = "retrieval-bound"
    next_step = ("Retrieval is the binding constraint. Invest the remaining "
                 "time in Experiments B (reranker fix) and C (BM25 × reranker fusion). "
                 "Experiment D (LLM cascade) is still worth running for novelty.")
elif delta < 0.05:
    interpretation = "classification-bound"
    next_step = ("Classification is the binding constraint — even with perfect "
                 "evidence the model cannot separate the four classes. Invest "
                 "the remaining time in Experiment D (LLM cascade for DISPUTED), "
                 "where the LLM addresses a class fine-tuning cannot. "
                 "Experiments B and C will not move the headline metric.")
else:
    interpretation = "mixed"
    next_step = ("The bottleneck is mixed — both retrieval and classification "
                 "are losing meaningful points. Run all three remaining experiments; "
                 "expect gains to compound rather than dominate.")

print(f"\nInterpretation: {interpretation}")
print(f"Recommended next step: {next_step}")

# Persist for the report.
with open(V10_DIR / "exp_A_oracle.json", "w", encoding="utf-8") as f:
    json.dump({
        "A_majority": A_majority, "A_v9": A_v9, "A_oracle": A_oracle,
        "delta_oracle_vs_v9": delta,
        "per_class_oracle": oracle_per_class.to_dict(orient="records"),
        "interpretation": interpretation, "next_step": next_step,
    }, f, indent=2)
print(f"\nWrote {V10_DIR / 'exp_A_oracle.json'}")

Per-class accuracy under oracle retrieval:
          label  n  accuracy
       SUPPORTS 68  0.720588
        REFUTES 27  0.000000
NOT_ENOUGH_INFO 41  0.292683
       DISPUTED 18  0.111111

Interpretation: classification-bound
Recommended next step: Classification is the binding constraint — even with perfect evidence the model cannot separate the four classes. Invest the remaining time in Experiment D (LLM cascade for DISPUTED), where the LLM addresses a class fine-tuning cannot. Experiments B and C will not move the headline metric.

Wrote outputs_notebook_v10\exp_A_oracle.json


## Experiment B — Reranker fix: pos_weight + lower learning rate

The v9 reranker training history is unusual. With BCE loss on a candidate pool that has four hard negatives for every positive, the model peaked at epoch 1 with dev F = 0.199 and then *decreased* monotonically to F = 0.178 by epoch 3. Training loss kept falling. That pattern — train loss down, dev metric down — is the classic signature of a model that is overconfidently pushing the easy negatives further from the decision boundary at the cost of its calibration on the hard examples that actually decide F-score.

The cheapest fix is two-part. First, set `pos_weight = 4.0` in `BCEWithLogitsLoss` so the positive-class loss is up-weighted to match the 4:1 negative-to-positive ratio in each batch; without this, the dominant gradient signal comes from "make these easy negatives even more negative" rather than "rank this positive above its hard negatives". Second, halve the learning rate from 1e-5 to 5e-6 so that even if pos_weight introduces some training instability, the optimiser does not overshoot.

We retrain a single reranker with both changes applied. If the new model's dev F at any epoch beats v9's best (F = 0.199), we adopt it as the v10 reranker. Otherwise the hypothesis is falsified, we keep v9's reranker, and we report the negative result honestly — itself useful, since it documents that we tried.

In [8]:
# Build reranker training pairs: gold positives + 4 hard negatives sampled
# from each claim's BM25 top-500 (identical to v9's procedure).
NEGATIVES_PER_POSITIVE = 4
RERANKER_BATCH_TRAIN = 32
RERANKER_BATCH_EVAL  = 128
RERANKER_EPOCHS_V10  = 3
RERANKER_LR_V10      = 5e-6              # halved from v9's 1e-5
RERANKER_POS_WEIGHT  = 4.0               # matches neg/pos ratio in the batch

class RerankerPairDataset(Dataset):
    def __init__(self, examples, claims_dict, evidence_dict, tokenizer,
                 max_len=RERANKER_MAX_LEN):
        self.examples = examples
        self.claims, self.evidence = claims_dict, evidence_dict
        self.tokenizer, self.max_len = tokenizer, max_len
    def __len__(self): return len(self.examples)
    def __getitem__(self, i):
        ex = self.examples[i]
        enc = self.tokenizer(
            self.claims[ex["cid"]]["claim_text"], self.evidence[ex["eid"]],
            truncation=True, padding="max_length", max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(ex["label"], dtype=torch.float32)
        return item


def build_reranker_examples(claims_dict, bm25_candidates, neg_per_pos=NEGATIVES_PER_POSITIVE):
    rng = random.Random(SEED)
    out = []
    for cid, claim in claims_dict.items():
        gold = [eid for eid in claim["evidences"] if eid in evidence]
        gold_set = set(gold)
        for eid in gold:
            out.append({"cid": cid, "eid": eid, "label": 1.0})
        cand_negs = [eid for eid, _ in bm25_candidates[cid][:BM25_CANDIDATE_K]
                     if eid not in gold_set and eid in evidence]
        needed = neg_per_pos * max(1, len(gold))
        if len(cand_negs) > needed:
            cand_negs = rng.sample(cand_negs, needed)
        for eid in cand_negs:
            out.append({"cid": cid, "eid": eid, "label": 0.0})
    rng.shuffle(out)
    return out

reranker_examples = build_reranker_examples(train_claims, train_bm25)
n_pos = sum(1 for e in reranker_examples if e["label"] == 1.0)
print(f"Reranker pairs: {len(reranker_examples):,}  "
      f"pos={n_pos:,}  neg={len(reranker_examples)-n_pos:,}  "
      f"(should match v9: 20,610 pos=4,122 neg=16,488)")

Reranker pairs: 20,610  pos=4,122  neg=16,488  (should match v9: 20,610 pos=4,122 neg=16,488)


In [9]:
# Score a candidate pool with a (loaded) reranker. Identical to v9.cell-18
# but exposed as a function we call multiple times below.
@torch.inference_mode()
def score_with_reranker(model, tokenizer, claims_dict, bm25_cands, split_name="dev"):
    model.eval()
    cache = {}
    for cid, claim in tqdm(list(claims_dict.items()), desc=f"Scoring {split_name}"):
        cand_eids = [eid for eid, _ in bm25_cands[cid]]
        bm25_scores = np.array([s for _, s in bm25_cands[cid]], dtype=np.float32)
        logits_all = []
        for start in range(0, len(cand_eids), RERANKER_BATCH_EVAL):
            batch_eids = cand_eids[start:start + RERANKER_BATCH_EVAL]
            enc = tokenizer(
                [claim["claim_text"]] * len(batch_eids),
                [evidence[eid] for eid in batch_eids],
                truncation=True, padding=True, max_length=RERANKER_MAX_LEN,
                return_tensors="pt",
            ).to(DEVICE, non_blocking=True)
            with torch.autocast(device_type="cuda" if DEVICE.type=="cuda" else "cpu",
                                dtype=AUTOCAST_DTYPE, enabled=(DEVICE.type=="cuda")):
                logits = model(**enc).logits.squeeze(-1)
            logits_all.extend(logits.float().detach().cpu().tolist())
        logits_np = np.asarray(logits_all, dtype=np.float32)
        cache[cid] = {
            "eids": cand_eids,
            "bm25": bm25_scores,
            "ce_logit": logits_np,
            "ce_prob":  1.0 / (1.0 + np.exp(-logits_np)),
        }
    return cache

In [10]:
# Train the fixed reranker. ~6 min/epoch on RTX 4060; total ~18 min.
print("Loading reranker base:", RERANKER_MODEL_NAME)
reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL_NAME)
reranker_model = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_MODEL_NAME, num_labels=1, ignore_mismatched_sizes=True,
).to(DEVICE)

reranker_loader = DataLoader(
    RerankerPairDataset(reranker_examples, train_claims, evidence, reranker_tokenizer),
    batch_size=RERANKER_BATCH_TRAIN, shuffle=True,
    pin_memory=(DEVICE.type=="cuda"), num_workers=0,
)

# pos_weight=4 inside BCEWithLogitsLoss: this scales the loss on positive
# examples by 4×, counterbalancing the 4× over-representation of negatives.
loss_fn_b = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([RERANKER_POS_WEIGHT], device=DEVICE)
)
optim_b = torch.optim.AdamW(reranker_model.parameters(),
                              lr=RERANKER_LR_V10, weight_decay=WEIGHT_DECAY)
total_steps_b = len(reranker_loader) * RERANKER_EPOCHS_V10
sched_b = get_linear_schedule_with_warmup(
    optim_b, num_warmup_steps=int(WARMUP_RATIO * total_steps_b),
    num_training_steps=total_steps_b,
)
scaler_b = (torch.amp.GradScaler("cuda", enabled=USE_GRAD_SCALER)
            if hasattr(torch.amp, "GradScaler")
            else torch.cuda.amp.GradScaler(enabled=USE_GRAD_SCALER))

# Helper: quick dev F via relative-δ at α=1 (no fusion yet), δ=0.25 (v9's epoch metric).
def quick_dev_F(model):
    cache = score_with_reranker(model, reranker_tokenizer, dev_claims, dev_bm25, "dev_quick")
    retr  = select_relative_delta(cache, delta=0.25)
    return evaluate_retrieval_only(retr, dev_claims), cache

best_F_b, best_state_b, best_cache_b, best_epoch_b = -1.0, None, None, -1
history_b = []
for epoch in range(1, RERANKER_EPOCHS_V10 + 1):
    reranker_model.train(); losses = []; t0 = time.time()
    for batch in tqdm(reranker_loader, desc=f"Reranker-v10 epoch {epoch}/{RERANKER_EPOCHS_V10}"):
        labels = batch.pop("labels").to(DEVICE, non_blocking=True)
        batch  = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        with torch.autocast(device_type="cuda" if DEVICE.type=="cuda" else "cpu",
                            dtype=AUTOCAST_DTYPE, enabled=(DEVICE.type=="cuda")):
            logits = reranker_model(**batch).logits.squeeze(-1)
            loss   = loss_fn_b(logits, labels)
        if USE_GRAD_SCALER:
            scaler_b.scale(loss).backward(); scaler_b.unscale_(optim_b)
            torch.nn.utils.clip_grad_norm_(reranker_model.parameters(), GRAD_CLIP)
            scaler_b.step(optim_b); scaler_b.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(reranker_model.parameters(), GRAD_CLIP)
            optim_b.step()
        sched_b.step(); optim_b.zero_grad(set_to_none=True)
        losses.append(float(loss.item()))

    F_dev, dev_cache = quick_dev_F(reranker_model)
    elapsed = time.time() - t0
    print(f"Epoch {epoch}: loss={np.mean(losses):.4f}  dev_F(δ=0.25)={F_dev:.4f}  "
          f"time={elapsed:.1f}s")
    history_b.append({"epoch": epoch, "loss": float(np.mean(losses)),
                       "dev_F": F_dev, "time_s": round(elapsed, 1)})
    if F_dev > best_F_b:
        best_F_b = F_dev; best_epoch_b = epoch
        best_state_b = {k: v.detach().clone().cpu() for k, v in reranker_model.state_dict().items()}
        best_cache_b = dev_cache

print(f"\nBest v10 reranker epoch = {best_epoch_b}   dev_F = {best_F_b:.4f}")
print(f"v9 reranker best (for reference): F = 0.1987 at epoch 1")
torch.save(best_state_b, V10_MODELS / "reranker_v10.pt")
pd.DataFrame(history_b)

Loading reranker base: cross-encoder/ms-marco-MiniLM-L-12-v2


Scoring dev_quick: 100%|██████████| 154/154 [00:56<00:00,  2.70it/s]


Epoch 1: loss=1.1086  dev_F(δ=0.25)=0.1814  time=148.1s


Scoring dev_quick: 100%|██████████| 154/154 [00:54<00:00,  2.84it/s]


Epoch 2: loss=0.6185  dev_F(δ=0.25)=0.1998  time=146.8s


Scoring dev_quick: 100%|██████████| 154/154 [00:55<00:00,  2.76it/s]

Epoch 3: loss=0.5641  dev_F(δ=0.25)=0.1937  time=149.7s

Best v10 reranker epoch = 2   dev_F = 0.1998
v9 reranker best (for reference): F = 0.1987 at epoch 1


,epoch,loss,dev_F,time_s
0,1,1.108581,0.181375,148.1
1,2,0.618480,0.199835,146.8
2,3,0.564073,0.193697,149.7


In [11]:
# Compare v9 vs v10 reranker side by side under each selection strategy on dev.
# This decides which reranker we use for Experiment C and onwards.
v10_better = best_F_b > 0.1987

print(f"v10 reranker {'BEATS' if v10_better else 'DOES NOT beat'} v9's best dev F.")
if v10_better:
    print("Adopting v10 reranker for downstream experiments. Re-scoring train/test ...")
    reranker_model.load_state_dict(best_state_b)
    train_ce = score_with_reranker(reranker_model, reranker_tokenizer,
                                     train_claims, train_bm25, "train")
    dev_ce   = best_cache_b
    test_ce  = score_with_reranker(reranker_model, reranker_tokenizer,
                                     test_claims, test_bm25, "test")
    for split, obj in [("train", train_ce), ("dev", dev_ce), ("test", test_ce)]:
        with open(V10_CACHE / f"{split}_ce_scores.pkl", "wb") as f:
            pickle.dump(obj, f)
    print("Cached new CE scores under outputs_notebook_v10/cache/.")
else:
    print("Keeping v9 reranker for downstream experiments.")
    train_ce, dev_ce, test_ce = train_ce_v9, dev_ce_v9, test_ce_v9

# Free reranker model to recover VRAM before later stages.
del reranker_model
gc.collect()
if DEVICE.type == "cuda": torch.cuda.empty_cache()

with open(V10_DIR / "exp_B_reranker.json", "w", encoding="utf-8") as f:
    json.dump({
        "v9_best_F": 0.1987, "v10_best_F": best_F_b,
        "v10_best_epoch": best_epoch_b, "v10_better": v10_better,
        "history": history_b,
        "hyperparams": {"lr": RERANKER_LR_V10, "pos_weight": RERANKER_POS_WEIGHT,
                         "epochs": RERANKER_EPOCHS_V10},
    }, f, indent=2)
print(f"\nWrote {V10_DIR / 'exp_B_reranker.json'}")

v10 reranker BEATS v9's best dev F.
Adopting v10 reranker for downstream experiments. Re-scoring train/test ...


Scoring test: 100%|██████████| 153/153 [00:56<00:00,  2.71it/s]


Cached new CE scores under outputs_notebook_v10/cache/.

Wrote outputs_notebook_v10\exp_B_reranker.json


## Experiment C — BM25 × Reranker score fusion

The reranker's cross-encoder logit and BM25's lexical score capture complementary signals. BM25 is well-calibrated within a query — its top-1 is the lexically closest evidence sentence to the claim — but it cannot tell paraphrase from coincidence. The reranker is the opposite: strong on semantic similarity but its logits are not calibrated across queries (a logit of 5 means "rank-1 for this claim", not "this evidence is true"). A common, well-supported trick on noisy retrieval is to combine them through a per-query-normalised weighted sum, letting the lexical anchor pull back the reranker's overconfident misses.

We min-max normalise each claim's BM25 scores into [0, 1] (since BM25 logits are unbounded and not directly comparable to the reranker's sigmoid prob), then compute `fused = α·ce_prob + (1−α)·bm25_norm` for α in a small grid. The reranker prob is already in [0, 1], so the two terms live on the same scale. We then re-run v9's relative-δ selector on the fused score, sweeping (α, δ) jointly on dev. The winning (α, δ) is locked in for the final config.

In [12]:
# Build a copy of the CE cache with an added "fused" score key. We also
# add a min-max normalised BM25 score per claim so the fusion is well-scaled.
def add_fused_scores(ce_cache, alpha):
    out = {}
    for cid, d in ce_cache.items():
        bm = d["bm25"].astype(np.float32)
        bm_min, bm_max = bm.min(), bm.max()
        bm_norm = (bm - bm_min) / max(bm_max - bm_min, 1e-9)   # per-claim min-max
        ce_p = d["ce_prob"].astype(np.float32)
        fused = alpha * ce_p + (1.0 - alpha) * bm_norm
        out[cid] = {**d, "bm25_norm": bm_norm, "fused": fused}
    return out

ALPHA_GRID = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
DELTA_GRID = [0.25, 0.50, 0.75, 1.0, 1.5, 2.0]

stage_c_rows = []
for alpha in ALPHA_GRID:
    fused_cache = add_fused_scores(dev_ce, alpha)
    for delta in DELTA_GRID:
        retr = select_relative_delta(fused_cache, delta=delta, score_key="fused")
        F = evaluate_retrieval_only(retr, dev_claims)
        avg = float(np.mean([len(v) for v in retr.values()]))
        stage_c_rows.append({"alpha": alpha, "delta": delta,
                              "dev_F": F, "avg_pred": avg})

stage_c_df = pd.DataFrame(stage_c_rows).sort_values("dev_F", ascending=False).reset_index(drop=True)
print("Top 10 (α, δ) combinations on dev:")
print(stage_c_df.head(10).to_string(index=False))

Top 10 (α, δ) combinations on dev:
 alpha  delta    dev_F  avg_pred
   0.8   0.25 0.218960  4.935065
   0.8   1.50 0.217136  5.000000
   0.8   0.75 0.217136  5.000000
   0.8   1.00 0.217136  5.000000
   0.8   0.50 0.217136  5.000000
   0.8   2.00 0.217136  5.000000
   0.9   0.25 0.216285  4.987013
   0.9   0.50 0.215636  5.000000
   0.9   0.75 0.215636  5.000000
   0.9   1.00 0.215636  5.000000


In [13]:
# Lock in the best (α, δ). If fusion doesn't help (best at α=1.0), we just
# fall back to pure reranker which is v9's behaviour anyway — still reportable.
best_c = stage_c_df.iloc[0]
BEST_ALPHA = float(best_c["alpha"])
BEST_DELTA = float(best_c["delta"])
print(f"BEST FUSION:  α = {BEST_ALPHA}   δ = {BEST_DELTA}")
print(f"  dev F = {best_c['dev_F']:.4f}   avg #pred = {best_c['avg_pred']:.2f}")
print(f"  vs v9 dev F (α=1, δ=1.5)  = 0.2448")

# Apply the winning config to all splits → final retrieval dicts for v10.
def fused_select(ce_cache):
    return select_relative_delta(add_fused_scores(ce_cache, BEST_ALPHA),
                                   delta=BEST_DELTA, score_key="fused")

train_retrieval = fused_select(train_ce)
dev_retrieval   = fused_select(dev_ce)
test_retrieval  = fused_select(test_ce)

for split, obj in [("train", train_retrieval), ("dev", dev_retrieval), ("test", test_retrieval)]:
    with open(V10_CACHE / f"{split}_final_evidence.pkl", "wb") as f:
        pickle.dump(obj, f)

final_dev_F = evaluate_retrieval_only(dev_retrieval, dev_claims)
print(f"\nFinal v10 dev retrieval F = {final_dev_F:.4f}")

with open(V10_DIR / "exp_C_fusion.json", "w", encoding="utf-8") as f:
    json.dump({
        "best_alpha": BEST_ALPHA, "best_delta": BEST_DELTA,
        "dev_F_v10": final_dev_F, "dev_F_v9": 0.2448,
        "improvement": final_dev_F - 0.2448,
        "grid": stage_c_df.to_dict(orient="records"),
    }, f, indent=2)
print(f"Wrote {V10_DIR / 'exp_C_fusion.json'}")

BEST FUSION:  α = 0.8   δ = 0.25
  dev F = 0.2190   avg #pred = 4.94
  vs v9 dev F (α=1, δ=1.5)  = 0.2448

Final v10 dev retrieval F = 0.2190
Wrote outputs_notebook_v10\exp_C_fusion.json


## Experiment D — LLM cascade for DISPUTED detection

DISPUTED is structurally the hardest class to learn from 1228 training examples. It has only 124 positives in train (10.1%), and unlike SUPPORTS / REFUTES the *signal* for DISPUTED is not "does this claim agree with this evidence" but "do the multiple retrieved evidence pieces disagree *with each other* about this claim". Standard fine-tuning sees each `(claim, concatenated-evidence)` as one i.i.d. example; the disagreement signal lives in the *relationship between evidence pieces*, which is hard to convey through the loss on so few examples.

This is exactly the kind of subproblem where a zero-shot LLM has an advantage: it does not need to learn the meta-relation from data, only follow a prompt that asks about it. We add an LLM as a **narrow cascade** layer: only for dev claims where the classifier predicts SUPPORTS or REFUTES with confidence above a threshold do we ask the LLM "do the retrieved evidence pieces *contradict each other* about this claim?" If yes, override to DISPUTED. The LLM never overrides DISPUTED predictions to anything else, and never speaks on NEI. This narrow, justifiable role is exactly the "intentional, designed LLM contribution" the rubric rewards over a "let the LLM do everything" approach.

We use **Qwen2.5-3B-Instruct** in 4-bit quantisation. The model is open-source (Apache 2.0), fits in ~3GB of VRAM, and runs on Colab's free tier. The cell below tries to load it; if `bitsandbytes` isn't installed, it falls back to bf16 (~6GB VRAM, still fits 4060 8GB but tight); if both fail, it skips this experiment and the rest of the notebook still runs.

In [14]:
# Try to load Qwen2.5-3B-Instruct in 4bit. Graceful degradation if libs absent.
LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"
llm_model, llm_tokenizer, llm_mode = None, None, None

try:
    from transformers import AutoTokenizer as _Tok, AutoModelForCausalLM as _CausalLM
    try:
        from transformers import BitsAndBytesConfig
        import bitsandbytes  # noqa: F401  (just to confirm installed)
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4",
        )
        print(f"Loading {LLM_NAME} in 4-bit ...")
        llm_tokenizer = _Tok.from_pretrained(LLM_NAME)
        llm_model = _CausalLM.from_pretrained(
            LLM_NAME, quantization_config=bnb_cfg, device_map="auto",
        )
        llm_mode = "4bit"
    except (ImportError, Exception) as e_4bit:
        print(f"4-bit unavailable ({type(e_4bit).__name__}: {e_4bit}); falling back to bf16.")
        print(f"To enable 4-bit next time: pip install bitsandbytes accelerate")
        llm_tokenizer = _Tok.from_pretrained(LLM_NAME)
        llm_model = _CausalLM.from_pretrained(
            LLM_NAME, torch_dtype=torch.bfloat16, device_map="auto",
        )
        llm_mode = "bf16"
    print(f"LLM loaded in {llm_mode} mode.")
except Exception as e:
    print(f"LLM load failed entirely ({type(e).__name__}: {e}).")
    print("Skipping Experiment D. Other experiments still run.")
    llm_model = None

4-bit unavailable (ModuleNotFoundError: No module named 'bitsandbytes'); falling back to bf16.
To enable 4-bit next time: pip install bitsandbytes accelerate


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:05<00:00, 81.94it/s] 


LLM loaded in bf16 mode.


In [15]:
# The LLM is asked a narrowly-scoped binary question.
# We deliberately avoid asking the LLM to do the four-way classification
# itself — its job here is only "do these evidence pieces contradict
# each other?", which is what fine-tuning cannot easily learn.

CONTRADICTION_PROMPT = (
    "You will be shown a CLAIM and several pieces of EVIDENCE retrieved for it.\n"
    "Your task: decide whether the evidence pieces contradict each other in their "
    "stance toward the CLAIM (i.e., some appear to support it while others appear "
    "to refute it, or they assert mutually incompatible facts).\n"
    "\nAnswer with a single word: YES or NO.\n"
    "Do NOT explain. Do NOT add anything else.\n"
    "\nCLAIM: {claim}\n"
    "\nEVIDENCE:\n{evidence_block}\n"
    "\nDo the evidence pieces contradict each other regarding the claim? Answer YES or NO."
)


@torch.inference_mode()
def llm_detects_contradiction(claim_text, ev_texts, max_new_tokens=4):
    """Returns True iff the LLM's first content token decodes to 'YES'."""
    evidence_block = "\n".join(f"[{i+1}] {t}" for i, t in enumerate(ev_texts))
    user_msg = CONTRADICTION_PROMPT.format(claim=claim_text, evidence_block=evidence_block)
    messages = [{"role": "user", "content": user_msg}]
    prompt = llm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = llm_tokenizer(prompt, return_tensors="pt").to(llm_model.device)
    out = llm_model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        temperature=1.0, top_p=1.0,
        pad_token_id=llm_tokenizer.eos_token_id,
    )
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    text = llm_tokenizer.decode(new_tokens, skip_special_tokens=True).strip().upper()
    return text.startswith("YES")


if llm_model is not None:
    # Smoke-test on one example before the full pass.
    cid_test, c_test = next(iter(dev_claims.items()))
    ev_test = [evidence[e] for e in dev_retrieval[cid_test][:3]]
    answer = llm_detects_contradiction(c_test["claim_text"], ev_test)
    print(f"Smoke test: claim={cid_test!r}  LLM contradiction answer = {answer}")
else:
    print("LLM unavailable; skipping smoke test.")

Smoke test: claim='claim-752'  LLM contradiction answer = False


In [16]:
# Run the cascade on dev. The base prediction comes from v9's classifier
# (loaded from disk if available, or freshly retrained). The LLM only
# overrides non-DISPUTED, non-NEI predictions where it detects contradiction.
def base_predictions_for_dev():
    """Use v9's classifier weights on v10's retrieval for the base layer."""
    tok = AutoTokenizer.from_pretrained(CLASSIFIER_MODEL_NAME)
    model = fresh_classifier()
    if v9_clf_path.exists():
        model.load_state_dict(torch.load(v9_clf_path, map_location=DEVICE))
        print("Base classifier: loaded v9 weights.")
    else:
        # Quick retrain. This is a fallback path only.
        del model; torch.cuda.empty_cache() if DEVICE.type=="cuda" else None
        best, _ = train_classifier(train_retrieval, dev_retrieval,
                                     weighting_power=0.0, epochs=CLASSIFIER_EPOCHS,
                                     tag="base-clf")
        model = fresh_classifier()
        model.load_state_dict(best["state"])
    ds, dl = make_loader(dev_claims, dev_retrieval, tok,
                          CLASSIFIER_BATCH_EVAL, include_gold=False, shuffle=False)
    preds = predict_labels(model, dl, ds)
    del model
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return preds

base_preds_dev = base_predictions_for_dev()
base_metrics = evaluate_submission(
    {cid: {"claim_label": base_preds_dev[cid], "evidences": dev_retrieval[cid]}
     for cid in dev_claims},
    dev_claims, verbose=False,
)
print(f"Base (no cascade): F={base_metrics['F']:.4f}  A={base_metrics['A']:.4f}  "
      f"H={base_metrics['H']:.4f}")

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1`.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4650.39it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1]) vs model:torch.Size([4])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1, 384]) vs model:torch.Size([4, 384])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Base classifier: loaded v9 weights.
Base (no cascade): F=0.2190  A=0.4935  H=0.3033


In [17]:
# Apply LLM cascade: ask only on SUPPORTS/REFUTES predictions (not NEI/DISPUTED).
# Rationale: NEI means "we have no evidence either way" — the LLM has no
# extra signal there. DISPUTED predictions are already what we'd route to;
# overriding them risks losing real positives.
if llm_model is not None:
    cascade_preds = dict(base_preds_dev)
    flipped = 0
    targets = [cid for cid in dev_claims
               if base_preds_dev[cid] in ("SUPPORTS", "REFUTES")
               and len(dev_retrieval[cid]) >= 2]    # need >=2 evidences for contradiction to be defined

    print(f"LLM cascade scoring {len(targets)} dev claims ...")
    for cid in tqdm(targets):
        c = dev_claims[cid]
        ev_texts = [evidence[e] for e in dev_retrieval[cid][:MAX_FINAL_K]]
        if llm_detects_contradiction(c["claim_text"], ev_texts):
            cascade_preds[cid] = "DISPUTED"
            flipped += 1

    cascade_metrics = evaluate_submission(
        {cid: {"claim_label": cascade_preds[cid], "evidences": dev_retrieval[cid]}
         for cid in dev_claims},
        dev_claims, verbose=False,
    )
    print(f"\nLLM cascade flipped {flipped}/{len(targets)} predictions to DISPUTED")
    print(f"With cascade:   F={cascade_metrics['F']:.4f}  A={cascade_metrics['A']:.4f}  "
          f"H={cascade_metrics['H']:.4f}")
    print(f"Without (base): F={base_metrics['F']:.4f}  A={base_metrics['A']:.4f}  "
          f"H={base_metrics['H']:.4f}")
    print(f"Delta A: {cascade_metrics['A'] - base_metrics['A']:+.4f}")

    # Per-class effect of the cascade.
    rows = []
    for lbl in LABELS:
        cids = [cid for cid, c in dev_claims.items() if c["claim_label"] == lbl]
        if not cids:
            rows.append({"label": lbl, "n": 0, "base_acc": 0.0, "cascade_acc": 0.0})
            continue
        base_acc = float(np.mean([base_preds_dev[cid] == lbl for cid in cids]))
        cas_acc  = float(np.mean([cascade_preds[cid] == lbl for cid in cids]))
        rows.append({"label": lbl, "n": len(cids),
                      "base_acc": base_acc, "cascade_acc": cas_acc,
                      "Δ": cas_acc - base_acc})
    cascade_per_class = pd.DataFrame(rows)
    print("\nPer-class accuracy with vs without cascade:")
    print(cascade_per_class.to_string(index=False))

    # Decide whether to keep the cascade. We keep it if A does not regress
    # by more than 0.01 and DISPUTED recall improves.
    A_regression = base_metrics['A'] - cascade_metrics['A']
    disputed_base = next(r['base_acc']    for r in rows if r['label'] == 'DISPUTED')
    disputed_cas  = next(r['cascade_acc'] for r in rows if r['label'] == 'DISPUTED')
    KEEP_CASCADE = (A_regression <= 0.01) and (disputed_cas > disputed_base)
    print(f"\nKeep cascade? {KEEP_CASCADE}  "
          f"(A_regression={A_regression:.4f}, DISPUTED Δ={disputed_cas-disputed_base:+.4f})")
else:
    cascade_preds = dict(base_preds_dev)
    cascade_metrics = dict(base_metrics)
    cascade_per_class = pd.DataFrame()
    KEEP_CASCADE = False
    print("Cascade skipped (no LLM); using base classifier predictions.")

with open(V10_DIR / "exp_D_cascade.json", "w", encoding="utf-8") as f:
    json.dump({
        "llm_mode": llm_mode, "llm_name": LLM_NAME if llm_model else None,
        "base_metrics": base_metrics, "cascade_metrics": cascade_metrics,
        "keep_cascade": bool(KEEP_CASCADE),
        "per_class": cascade_per_class.to_dict(orient="records") if len(cascade_per_class) else [],
    }, f, indent=2)
print(f"\nWrote {V10_DIR / 'exp_D_cascade.json'}")

# Free the LLM to recover VRAM for the rest of the notebook.
if llm_model is not None:
    del llm_model, llm_tokenizer
    gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

LLM cascade scoring 139 dev claims ...


100%|██████████| 139/139 [00:26<00:00,  5.27it/s]



LLM cascade flipped 0/139 predictions to DISPUTED
With cascade:   F=0.2190  A=0.4935  H=0.3033
Without (base): F=0.2190  A=0.4935  H=0.3033
Delta A: +0.0000

Per-class accuracy with vs without cascade:
          label  n  base_acc  cascade_acc   Δ
       SUPPORTS 68  0.970588     0.970588 0.0
        REFUTES 27  0.000000     0.000000 0.0
NOT_ENOUGH_INFO 41  0.121951     0.121951 0.0
       DISPUTED 18  0.277778     0.277778 0.0

Keep cascade? False  (A_regression=0.0000, DISPUTED Δ=+0.0000)

Wrote outputs_notebook_v10\exp_D_cascade.json


## Experiment E — Per-class error analysis for the report

The Critical Analysis section of the report needs qualitative material: not just "REFUTES recall is 0%", but *examples* of REFUTES claims the system mishandled, with an explanation of why. The cell below builds a per-claim DataFrame of dev errors and then pulls representative cases for each of three diagnostic buckets: (i) the retrieval missed the gold evidence entirely — these are pure retrieval failures and no classifier could fix them; (ii) retrieval found at least some gold evidence but the classifier chose the wrong label — these are pure classification failures; (iii) the system predicted the right label for the wrong reason (random retrieval, lucky classifier). The split into these three buckets is itself a defensible finding for the report.

In [18]:
# Decide which final prediction set we are analysing — cascade if kept, base otherwise.
final_pred_labels = cascade_preds if KEEP_CASCADE else base_preds_dev
final_retrieval   = dev_retrieval                # v10 retrieval (post-fusion)

# Build per-claim diagnostic frame.
rows = []
for cid, c in dev_claims.items():
    gold_set  = set(c["evidences"])
    pred_eids = final_retrieval[cid]
    pred_set  = set(pred_eids)
    correct_ev = gold_set & pred_set
    retrieval_F = evidence_f1_for_claim(pred_eids, c["evidences"])
    label_correct = final_pred_labels[cid] == c["claim_label"]
    rows.append({
        "cid": cid,
        "claim": c["claim_text"],
        "gold_label": c["claim_label"],
        "pred_label": final_pred_labels[cid],
        "label_correct": label_correct,
        "n_gold_ev": len(gold_set),
        "n_pred_ev": len(pred_eids),
        "n_correct_ev": len(correct_ev),
        "retrieval_F": retrieval_F,
        "gold_eids": list(gold_set),
        "pred_eids": list(pred_eids),
    })
err_df = pd.DataFrame(rows)

# Bucket the failure modes.
err_df["bucket"] = "all_correct"
err_df.loc[(~err_df["label_correct"]) & (err_df["n_correct_ev"] == 0), "bucket"] = "retrieval_failure"
err_df.loc[(~err_df["label_correct"]) & (err_df["n_correct_ev"] > 0),  "bucket"] = "classification_failure"
err_df.loc[(err_df["label_correct"])  & (err_df["n_correct_ev"] == 0), "bucket"] = "lucky_classification"

bucket_summary = err_df.groupby("bucket").agg(
    n=("cid", "size"),
    mean_retrieval_F=("retrieval_F", "mean"),
).reset_index()
print("Failure mode summary on dev:")
print(bucket_summary.to_string(index=False))

Failure mode summary on dev:
                bucket  n  mean_retrieval_F
           all_correct 47          0.431189
classification_failure 40          0.336349
  lucky_classification 29          0.000000
     retrieval_failure 38          0.000000


In [19]:
# Confusion matrix + per-class table for the report.
def confusion_matrix_df(gold_claims, pred_labels):
    mat = pd.DataFrame(0, index=LABELS, columns=LABELS)
    for cid, c in gold_claims.items():
        mat.loc[c["claim_label"], pred_labels[cid]] += 1
    return mat

print("Confusion matrix (rows=gold, cols=pred):")
cm_df = confusion_matrix_df(dev_claims, final_pred_labels)
print(cm_df.to_string())
cm_df.to_csv(V10_DIR / "confusion_matrix.csv")

per_class_rows = []
for lbl in LABELS:
    cids_lbl = [cid for cid, c in dev_claims.items() if c["claim_label"] == lbl]
    acc = float(np.mean([final_pred_labels[cid] == lbl for cid in cids_lbl])) if cids_lbl else 0.0
    rF  = float(np.mean([evidence_f1_for_claim(final_retrieval[cid], dev_claims[cid]["evidences"])
                          for cid in cids_lbl])) if cids_lbl else 0.0
    per_class_rows.append({"label": lbl, "n": len(cids_lbl),
                            "class_accuracy": acc, "retrieval_F": rF})
per_class_df = pd.DataFrame(per_class_rows)
print("\nPer-class diagnostics:")
print(per_class_df.to_string(index=False))
per_class_df.to_csv(V10_DIR / "per_class_diagnostics.csv", index=False)

Confusion matrix (rows=gold, cols=pred):
                 SUPPORTS  REFUTES  NOT_ENOUGH_INFO  DISPUTED
SUPPORTS               66        0                1         1
REFUTES                25        0                2         0
NOT_ENOUGH_INFO        35        0                5         1
DISPUTED               13        0                0         5

Per-class diagnostics:
          label  n  class_accuracy  retrieval_F
       SUPPORTS 68        0.970588     0.256501
        REFUTES 27        0.000000     0.071576
NOT_ENOUGH_INFO 41        0.121951     0.212195
       DISPUTED 18        0.277778     0.313624


In [20]:
# Sample 2 examples from each error bucket for the report's qualitative section.
# Deterministic ordering: seed-fixed sample, sorted by cid for stability.
def sample_errors(bucket, k=2):
    sub = err_df[err_df["bucket"] == bucket].sort_values("cid")
    return sub.head(k)

print("=" * 70)
print("QUALITATIVE ERROR ANALYSIS — paste into report's Critical Analysis section")
print("=" * 70)

for bucket_name in ["retrieval_failure", "classification_failure", "lucky_classification"]:
    sub = sample_errors(bucket_name, k=2)
    print(f"\n--- {bucket_name.upper()} ({len(sub)} example(s) shown) ---")
    for _, row in sub.iterrows():
        print(f"\n  cid: {row['cid']}")
        print(f"  claim: {row['claim'][:200]}")
        print(f"  gold label: {row['gold_label']}   pred label: {row['pred_label']}")
        print(f"  retrieval F: {row['retrieval_F']:.3f}  "
              f"(found {row['n_correct_ev']}/{row['n_gold_ev']} gold evidences)")
        print(f"  gold evidences ({len(row['gold_eids'])}):")
        for eid in row['gold_eids'][:3]:
            t = evidence[eid][:160].replace("\n", " ")
            print(f"    [{eid}] {t}")
        print(f"  predicted evidences ({len(row['pred_eids'])}):")
        for eid in row['pred_eids'][:3]:
            mark = "✓" if eid in row['gold_eids'] else " "
            t = evidence[eid][:160].replace("\n", " ")
            print(f"    {mark} [{eid}] {t}")

# Save full DataFrame for further analysis in the report.
err_df.drop(columns=["gold_eids", "pred_eids"]).to_csv(
    V10_DIR / "per_claim_dev_diagnostics.csv", index=False)
print(f"\nWrote {V10_DIR / 'per_claim_dev_diagnostics.csv'} (one row per dev claim)")

QUALITATIVE ERROR ANALYSIS — paste into report's Critical Analysis section

--- RETRIEVAL_FAILURE (2 example(s) shown) ---

  cid: claim-1292
  claim: Any reasonable person can recognize both positives and negatives among the policy proposals of both Tories and Labour.
  gold label: NOT_ENOUGH_INFO   pred label: SUPPORTS
  retrieval F: 0.000  (found 0/5 gold evidences)
  gold evidences (5):
    [evidence-342343] Less attention was given to policy areas that might have been problematic for the Conservatives, like the NHS or housing (policy topics favoured by Labour) or i
    [evidence-462075] The Liberal Democrats, the Greens, the SNP and Labour all support a ban on fracking, whilst the Conservatives propose approving fracking on a case-by-case basis
    [evidence-399707] In a speech in Tynemouth the next day, May said Labour had "deserted" working-class voters, criticised Labour's policy proposals and said Britain's future depen
  predicted evidences (5):
      [evidence-656122] The fl

## Experiment F — Seed variance on the final configuration

A single-seed point estimate is fine for ablation but it is uninformative for the headline metric. Three seeds is the minimum to compute a meaningful standard deviation. We retrain only the classifier — the reranker and selection are deterministic given the data — under three seeds and report mean ± std on dev F, A, H. The report can then state "F = 0.244 ± 0.005, A = 0.481 ± 0.012, H = 0.323 ± 0.007" instead of single numbers, which is both more honest and a more defensible result. Three seeds at ~1 minute each is a small cost for a meaningful improvement in scientific rigour.

In [21]:
# Three seeds, classifier-only retraining. The retrieval (BM25 → reranker
# → fusion → relative-δ) is deterministic given the data, so we hold it
# fixed and only vary the classifier seed. This isolates the classifier's
# contribution to variance.
SEEDS = [42, 123, 2024]
variance_runs = []
for s in SEEDS:
    print(f"\n--- Seed {s} ---")
    best, _ = train_classifier(
        train_retrieval, dev_retrieval,
        weighting_power=0.0, epochs=CLASSIFIER_EPOCHS,
        seed=s, tag=f"final-s{s}",
    )
    # Optionally apply the cascade we chose in Experiment D.
    preds = dict(best["pred_labels"])
    variance_runs.append({"seed": s, "F": best["F"], "A": best["A"], "H": best["H"],
                          "pred_labels": preds})

var_df = pd.DataFrame([{k: r[k] for k in ("seed", "F", "A", "H")} for r in variance_runs])
print("\nPer-seed final metrics:")
print(var_df.to_string(index=False))
print("\nMean ± std:")
print(f"  F = {var_df['F'].mean():.4f} ± {var_df['F'].std(ddof=1):.4f}")
print(f"  A = {var_df['A'].mean():.4f} ± {var_df['A'].std(ddof=1):.4f}")
print(f"  H = {var_df['H'].mean():.4f} ± {var_df['H'].std(ddof=1):.4f}")

with open(V10_DIR / "exp_F_variance.json", "w", encoding="utf-8") as f:
    json.dump({
        "seeds": SEEDS,
        "per_seed": [{k: r[k] for k in ("seed", "F", "A", "H")} for r in variance_runs],
        "mean": {"F": float(var_df["F"].mean()), "A": float(var_df["A"].mean()), "H": float(var_df["H"].mean())},
        "std":  {"F": float(var_df["F"].std(ddof=1)), "A": float(var_df["A"].std(ddof=1)),
                  "H": float(var_df["H"].std(ddof=1))},
    }, f, indent=2)
print(f"\nWrote {V10_DIR / 'exp_F_variance.json'}")


--- Seed 42 ---


[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1`.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4681.74it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1]) vs model:torch.Size([4])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1, 384]) vs model:torch.Size([4, 384])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  final-s42 ep1: loss=1.2753  F=0.2190  A=0.4351  H=0.2913  (14.7s)


  final-s42 ep2: loss=1.1880  F=0.2190  A=0.4675  H=0.2982  (14.4s)


  final-s42 ep3: loss=1.1209  F=0.2190  A=0.4740  H=0.2996  (14.4s)


...

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1`.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6593.99it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1]) vs model:torch.Size([4])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1, 384]) vs model:torch.Size([4, 384])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  final-s2024 ep1: loss=1.3524  F=0.2190  A=0.4416  H=0.2928  (13.9s)


  final-s2024 ep2: loss=1.2824  F=0.2190  A=0.4351  H=0.2913  (13.8s)


  final-s2024 ep3: loss=1.2379  F=0.2190  A=0.4610  H=0.2969  (13.8s)


  final-s2024 ep4: loss=1.1980  F=0.2190  A=0.4610  H=0.2969  (13.8s)
  best ep3: F=0.2190 A=0.4610 H=0.2969

Per-seed final metrics:
 seed       F        A        H
   42 0.21896 0.493506 0.303335
  123 0.21896 0.487013 0.302098
 2024 0.21896 0.461039 0.296910

Mean ± std:
  F = 0.2190 ± 0.0000
  A = 0.4805 ± 0.0172
  H = 0.3008 ± 0.0034

Wrote outputs_notebook_v10\exp_F_variance.json


## Final summary

The table below collects every result you need for the Results section of the report: the v9 baseline, each of the five v10 experiments, and the seed-averaged final configuration. The summary JSON written at the end is ready to be loaded into the report's analysis.

In [22]:
# Headline comparison table for the report.
summary_rows = [
    {"system": "v9 baseline",                          "F": 0.2448, "A": 0.4805, "H": 0.3243,
     "notes": "BM25 PP3 R-both → MiniLM-marco rerank → relative-δ@1.5 → MiniLM-marco classifier W0"},
    {"system": "v9 + Oracle retrieval (Exp A)",        "F": 1.0,    "A": A_oracle,
     "H": 2*1.0*A_oracle/(1.0+A_oracle) if A_oracle > 0 else 0.0,
     "notes": "Diagnostic — feeds gold evidence to v9 classifier; not a real system"},
    {"system": "v10: fixed reranker (Exp B)",
     "F": best_F_b, "A": float("nan"), "H": float("nan"),
     "notes": f"BCE pos_weight=4, lr=5e-6, 3 epochs; best epoch={best_epoch_b}"},
    {"system": "v10: + BM25×reranker fusion (Exp C)",
     "F": final_dev_F, "A": float("nan"), "H": float("nan"),
     "notes": f"α={BEST_ALPHA}, δ={BEST_DELTA}"},
    {"system": "v10: + LLM cascade (Exp D)",
     "F": cascade_metrics["F"], "A": cascade_metrics["A"], "H": cascade_metrics["H"],
     "notes": f"Cascade {'kept' if KEEP_CASCADE else 'dropped'}; "
              f"LLM={LLM_NAME if llm_model is not None else 'unavailable'}"},
    {"system": "v10: final, mean over 3 seeds (Exp F)",
     "F": float(var_df["F"].mean()), "A": float(var_df["A"].mean()),
     "H": float(var_df["H"].mean()),
     "notes": (f"std: F±{var_df['F'].std(ddof=1):.4f}, "
                f"A±{var_df['A'].std(ddof=1):.4f}, "
                f"H±{var_df['H'].std(ddof=1):.4f}")},
]
summary_df = pd.DataFrame(summary_rows)
print("FINAL SUMMARY")
print("=" * 90)
print(summary_df.to_string(index=False))
summary_df.to_csv(V10_DIR / "summary.csv", index=False)

with open(V10_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump({
        "v9_baseline":   {"F": 0.2448, "A": 0.4805, "H": 0.3243},
        "A_oracle":      A_oracle,
        "exp_B": {"v9_F": 0.1987, "v10_F": best_F_b, "kept": best_F_b > 0.1987},
        "exp_C": {"alpha": BEST_ALPHA, "delta": BEST_DELTA, "dev_F": final_dev_F},
        "exp_D": {"cascade_kept": bool(KEEP_CASCADE),
                   "base":    base_metrics,
                   "cascade": cascade_metrics},
        "exp_F": {"seeds": SEEDS,
                   "F_mean": float(var_df["F"].mean()), "F_std": float(var_df["F"].std(ddof=1)),
                   "A_mean": float(var_df["A"].mean()), "A_std": float(var_df["A"].std(ddof=1)),
                   "H_mean": float(var_df["H"].mean()), "H_std": float(var_df["H"].std(ddof=1))},
    }, f, indent=2)
print(f"\nAll v10 artifacts saved under {V10_DIR}/")
print("Read summary.json + per_claim_dev_diagnostics.csv for the report.")

NameError: name 'llm_model' is not defined